In [1]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.databricks:spark-xml_2.12:0.18.0",
        "spark.sql.legacy.timeParserPolicy": "LEGACY",
        "spark.sql.catalog.iceberg_catalog": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        "spark.sql.catalog.iceberg_catalog.catalog-impl": "org.apache.iceberg.aws.glue.GlueCatalog"
    }
}

In [2]:
from pyspark.sql import functions as f
from pyspark.sql.types import *


VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1772783249427_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
opco = "oh"
run_datetime = "20260219_213100"
part_date = "20260219_2131"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
source_paths = [
    "s3://aep-datalake-raw-dev/util/intervals/nonvee/uiq_info/oh/20260219_2131/20260219-8394453c-d45c-4040-b489-bca288e7c734-1-1.xml.gz",
    "s3://aep-datalake-raw-dev/util/intervals/nonvee/uiq_info/oh/20260219_2131/20260219-8394453c-d45c-4040-b489-bca288e7c734-1-10.xml.gz",
    "s3://aep-datalake-raw-dev/util/intervals/nonvee/uiq_info/oh/20260219_2131/20260219-8394453c-d45c-4040-b489-bca288e7c734-1-100.xml.gz"
]

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
paths = ",".join(source_paths)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
# ------------------------------------------------------------
# Reference: stg_nonvee.interval_data_files_oh_src.ddl
# DDL Line 10: interval_reading array<struct<endtime,blocksequencenumber,
#              gatewaycollectedtime,intervalsequencenumber,
#              `interval`:array<struct<channel,rawvalue,value,uom,blockendvalue>>>>
# ------------------------------------------------------------

info_xml_schema = StructType([
    StructField("_MeterName", StringType(), True),
    StructField("_UtilDeviceID", StringType(), True),
    StructField("_MacID", StringType(), True),
    
    StructField("IntervalReadData", StructType([
        StructField("_IntervalLength", StringType(), True),
        StructField("_StartTime", StringType(), True),
        StructField("_EndTime", StringType(), True),
        StructField("_NumberIntervals", StringType(), True),
        
        StructField("Interval", ArrayType(
            StructType([
                StructField("_EndTime", StringType(), True),
                StructField("_BlockSequenceNumber", StringType(), True),
                StructField("_GatewayCollectedTime", StringType(), True),
                StructField("_IntervalSequenceNumber", StringType(), True),
                
                StructField("Reading", ArrayType(
                    StructType([
                        StructField("_Channel", StringType(), True),
                        StructField("_RawValue", StringType(), True),
                        StructField("_Value", StringType(), True),
                        StructField("_UOM", StringType(), True),
                        StructField("_BlockEndValue", StringType(), True)
                    ]), True)
                )
            ]), True)
        )
    ]), True)
])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
interval_data_files_oh_src_df = (
    spark.read
    .format("com.databricks.spark.xml")
    .option("rowTag", "MeterData")
    .option("attributePrefix", "_")
    .option("valueTag", "_VALUE")
    .option("nullValue", "")
    .option("mode", "PERMISSIVE")
    .schema(info_xml_schema)
    .load(paths)
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
interval_data_files_oh_src_df = (
    interval_data_files_oh_src_df
    .withColumnRenamed("_MeterName", "metername")              # DDL Line 3
    .withColumnRenamed("_UtilDeviceID", "utildeviceid")        # DDL Line 4
    .withColumnRenamed("_MacID", "macid")                      # DDL Line 5
    .withColumnRenamed("IntervalReadData", "interval_reading") # DDL Line 10
    .withColumn("part_date", f.lit(part_date))                 # DDL Line 12
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
interval_data_files_oh_src_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- metername: string (nullable = true)
 |-- utildeviceid: string (nullable = true)
 |-- macid: string (nullable = true)
 |-- interval_reading: struct (nullable = true)
 |    |-- _IntervalLength: string (nullable = true)
 |    |-- _StartTime: string (nullable = true)
 |    |-- _EndTime: string (nullable = true)
 |    |-- _NumberIntervals: string (nullable = true)
 |    |-- Interval: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _EndTime: string (nullable = true)
 |    |    |    |-- _BlockSequenceNumber: string (nullable = true)
 |    |    |    |-- _GatewayCollectedTime: string (nullable = true)
 |    |    |    |-- _IntervalSequenceNumber: string (nullable = true)
 |    |    |    |-- Reading: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- _Channel: string (nullable = true)
 |    |    |    |    |    |-- _RawValue: string (nullable = true)
 |    |    |    |    |  

# Reference: stg_nonvee.interval_data_files_oh_src_vw

In [10]:
# ============================================================
# Reference: stg_nonvee.interval_data_files_oh_src_vw
# Source: scripts/ddl/stg_nonvee.interval_data_files_oh_src_vw.ddl
# Called by: scripts/source_interval_data_files.sh
# ============================================================
# 
# LOGIC:
# 1. 1st EXPLODE: Flatten interval_reading array → get int_endtime, int_reading, etc.
# 2. 2nd EXPLODE: Flatten int_reading array → get channel, rawvalue, value, uom
# 3. Time Calculations: starttime = endtime - (IntervalLength * 60 seconds)
# 4. Filter: Remove records where channel/rawvalue/value/uom is NULL
# 5. LEFT JOIN: Join with uom_mapping to get aep_derived_uom, name_register, etc.
#
# ============================================================

# ------------------------------------------------------------
# UOM Mapping Reference Table
# ------------------------------------------------------------
uom_data = [
    ("oh", "1", "KWH", "kwh", "kWh_del_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "2", "KW", "kw", "kW_del_mtr_ivl_len_min", "DEMAND", "N"),
    ("oh", "3", "KVARH", "kvarh", "kVArh_del_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "4", "KVAR", "kvar", "kVAr_del_mtr_ivl_len_min", "DEMAND", "N"),
    ("oh", "5", "KVAH", "kvah", "kVAh_del_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "100", "KWH", "wh", "kWh_del_mtr_ivl_len_min", "INTERVAL", "Y"),
    ("oh", "101", "KWH", "kwh", "kWh_del_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "102", "KWH", "kwh(rec)", "kWh_rec_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "108", "KVARH", "kvarh", "kVArh_del_mtr_ivl_len_min", "INTERVAL", "N"),
    ("oh", "109", "KVAH", "kvah", "kVAh_del_mtr_ivl_len_min", "INTERVAL", "N"),
]

uom_schema = StructType([
    StructField("aep_opco", StringType()),
    StructField("aep_channel_id", StringType()),
    StructField("aep_derived_uom", StringType()),
    StructField("aep_raw_uom", StringType()),
    StructField("name_register", StringType()),
    StructField("aep_srvc_qlty_idntfr", StringType()),
    StructField("value_mltplr_flg", StringType())
])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
uom_mapping_df = spark.createDataFrame(uom_data, uom_schema)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# ------------------------------------------------------------
# 1st EXPLODE: Flatten interval_reading array
# ------------------------------------------------------------
interval_exploded_df = (
    interval_data_files_oh_src_df
    .withColumn("filename", f.input_file_name())
    .withColumn("exp_interval", f.explode("interval_reading.Interval"))
    .select(
        f.col("filename"),
        f.col("metername").alias("MeterName"),
        f.col("utildeviceid").alias("UtilDeviceID"),
        f.col("macid").alias("MacID"),
        f.col("interval_reading._IntervalLength").alias("IntervalLength"),
        f.col("interval_reading._StartTime").alias("blockstarttime"),
        f.col("interval_reading._EndTime").alias("blockendtime"),
        f.col("interval_reading._NumberIntervals").alias("NumberIntervals"),
        f.col("part_date"),
        f.col("exp_interval._EndTime").alias("int_endtime"),
        f.col("exp_interval._GatewayCollectedTime").alias("int_gatewaycollectedtime"),
        f.col("exp_interval._BlockSequenceNumber").alias("int_blocksequencenumber"),
        f.col("exp_interval._IntervalSequenceNumber").alias("int_intervalsequencenumber"),
        f.col("exp_interval.Reading").alias("int_reading")
    )
    .filter(
        f.date_format(
            f.to_timestamp(f.substring("blockstarttime", 1, 10), "yyyy-MM-dd"),
            "yyyyMMdd"
        ).cast("int") >= 20150301
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# ------------------------------------------------------------
# 2nd EXPLODE: Flatten int_reading array + time calculations
# ------------------------------------------------------------
reading_exploded_df = (
    interval_exploded_df
    .withColumn("exp_reading", f.explode("int_reading"))
    .select(
        f.col("filename"),
        f.col("MeterName"),
        f.col("UtilDeviceID"),
        f.col("MacID"),
        f.col("IntervalLength"),
        f.col("blockstarttime"),
        f.col("blockendtime"),
        f.col("NumberIntervals"),
        f.from_unixtime(
            f.unix_timestamp(f.substring("int_endtime", 1, 19), "yyyy-MM-dd'T'HH:mm:ss")
            - (f.col("IntervalLength").cast("int") * 60),
            "yyyy-MM-dd'T'HH:mm:ss"
        ).alias("starttime"),
        f.from_unixtime(
            f.unix_timestamp(f.substring("int_endtime", 1, 19), "yyyy-MM-dd'T'HH:mm:ss"),
            "yyyy-MM-dd'T'HH:mm:ss"
        ).alias("endtime"),
        f.substring("int_endtime", -6, 6).alias("interval_epoch"),
        f.col("int_gatewaycollectedtime"),
        f.col("int_blocksequencenumber"),
        f.col("int_intervalsequencenumber"),
        f.col("exp_reading._Channel").alias("channel"),
        f.col("exp_reading._RawValue").alias("rawvalue"),
        f.col("exp_reading._Value").alias("value"),
        f.col("exp_reading._UOM").alias("uom"),
        f.col("part_date")
    )
    .filter(
        f.col("channel").isNotNull() &
        f.col("rawvalue").isNotNull() &
        f.col("value").isNotNull() &
        f.col("uom").isNotNull()
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# ------------------------------------------------------------
# LEFT JOIN with uom_mapping 
# ------------------------------------------------------------
interval_data_files_oh_src_vw_df = (
    reading_exploded_df
    .join(uom_mapping_df, reading_exploded_df.channel == uom_mapping_df.aep_channel_id, "left")
    .select(
        f.col("filename"),
        f.trim(f.col("MeterName")).alias("MeterName"),
        f.col("UtilDeviceID"),
        f.col("MacID"),
        f.col("IntervalLength"),
        f.col("blockstarttime"),
        f.col("blockendtime"),
        f.col("NumberIntervals"),
        f.concat(f.col("starttime"), f.col("interval_epoch")).alias("starttime"),
        f.concat(f.col("endtime"), f.col("interval_epoch")).alias("endtime"),
        f.col("interval_epoch"),
        f.col("int_gatewaycollectedtime"),
        f.col("int_blocksequencenumber"),
        f.col("int_intervalsequencenumber"),
        f.col("channel"),
        f.col("rawvalue"),
        f.col("value"),
        f.lower(f.trim(f.col("uom"))).alias("uom"),
        f.col("part_date"),
        f.coalesce(f.col("aep_derived_uom"), f.lit("UNK")).alias("aep_uom"),
        f.when(
            f.col("name_register").isNotNull(),
            f.regexp_replace(f.col("name_register"), "mtr_ivl_len", f.col("IntervalLength"))
        ).otherwise(f.lit("UNK")).alias("name_register"),
        f.coalesce(f.col("aep_srvc_qlty_idntfr"), f.lit("UNK")).alias("aep_sqi"),
        f.col("value_mltplr_flg")
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
interval_data_files_oh_src_vw_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- filename: string (nullable = false)
 |-- MeterName: string (nullable = true)
 |-- UtilDeviceID: string (nullable = true)
 |-- MacID: string (nullable = true)
 |-- IntervalLength: string (nullable = true)
 |-- blockstarttime: string (nullable = true)
 |-- blockendtime: string (nullable = true)
 |-- NumberIntervals: string (nullable = true)
 |-- starttime: string (nullable = true)
 |-- endtime: string (nullable = true)
 |-- interval_epoch: string (nullable = true)
 |-- int_gatewaycollectedtime: string (nullable = true)
 |-- int_blocksequencenumber: string (nullable = true)
 |-- int_intervalsequencenumber: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- rawvalue: string (nullable = true)
 |-- value: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- part_date: string (nullable = false)
 |-- aep_uom: string (nullable = false)
 |-- name_register: string (nullable = true)
 |-- aep_sqi: string (nullable = false)
 |-- value_mltplr_flg: string (nulla

In [16]:
# ============================================================
# Reference: stg_nonvee.interval_data_files_oh_stg
# Source: scripts/ddl/stg_nonvee.interval_data_files_oh_stg.ddl
#         scripts/stage_interval_xml_files.sh (Lines 194-272)
# ============================================================
#
# LOGIC:
# 1. Read from interval_data_files_oh_src_vw_df (Script 2)
# 2. Read MACS reference table from Glue catalog
# 3. LEFT JOIN on metername + time window validation
# 4. Rename/transform columns to match DDL (41 columns)
#
# ============================================================

# Parameters
VAR_OPCO = "oh"
HDP_UPDATE_USER = "info-insert"
target_companies = ["10", "07"]

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
# ------------------------------------------------------------
# MACS Reference Table (from Glue Catalog)
# ------------------------------------------------------------
macs_df = (
    spark.table("stg_nonvee.meter_premise_macs_ami")
    .filter(f.col("co_cd_ownr").isin(target_companies))
    .select(
        f.col("prem_nb"),
        f.col("bill_cnst"),
        f.col("acct_clas_cd"),
        f.col("acct_type_cd"),
        f.col("devc_cd"),
        f.col("pgm_id_nm"),
        f.concat(
            f.coalesce(f.col("tx_mand_data"), f.lit("")),
            f.coalesce(f.col("doe_nb"), f.lit("")),
            f.coalesce(f.col("serv_deliv_id"), f.lit(""))
        ).alias("sd"),
        f.col("mfr_devc_ser_nbr"),
        f.col("mtr_pnt_nb"),
        f.col("tarf_pnt_nb"),
        f.lit(None).cast(StringType()).alias("cmsg_mtr_mult_nb"),
        f.col("mtr_inst_ts"),
        f.col("mtr_rmvl_ts"),
        f.unix_timestamp("mtr_inst_ts", "yyyy-MM-dd HH:mm:ss").alias("unix_mtr_inst_ts"),
        f.when(
            f.col("mtr_rmvl_ts") == "9999-01-01",
            f.unix_timestamp("mtr_rmvl_ts", "yyyy-MM-dd")
        ).otherwise(
            f.unix_timestamp("mtr_rmvl_ts", "yyyy-MM-dd HH:mm:ss")
        ).alias("unix_mtr_rmvl_ts"),
        f.unix_timestamp("acct_turn_on_dt", "yyyy-MM-dd").alias("unix_acct_turn_on_dt"),
        f.unix_timestamp("acct_turn_off_dt", "yyyy-MM-dd").alias("unix_acct_turn_off_dt"),
        f.col("serv_city_ad").alias("aep_city"),
        f.col("serv_zip_ad").alias("aep_zip"),
        f.col("state_cd").alias("aep_state")
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# ------------------------------------------------------------
# LEFT JOIN with MACS + SELECT (41 columns)
# ------------------------------------------------------------
interval_data_files_oh_stg_df = (
        interval_data_files_oh_src_vw_df
    .withColumn("unix_endtime", f.unix_timestamp("endtime", "yyyy-MM-dd'T'HH:mm:ssXXX"))
    .join(
        macs_df,
        (f.col("MeterName") == f.col("mfr_devc_ser_nbr")) &
        (f.col("unix_endtime").between(f.col("unix_mtr_inst_ts"), f.col("unix_mtr_rmvl_ts"))) &
        (f.col("unix_endtime").between(f.col("unix_acct_turn_on_dt"), f.col("unix_acct_turn_off_dt"))),
        "left"
    )
    .select(
        f.col("filename"),
        f.lit(VAR_OPCO).alias("aep_opco"),
        f.col("MeterName").alias("serialnumber"),
        f.col("UtilDeviceID").alias("utildeviceid"),
        f.col("MacID").alias("macid"),
        f.col("IntervalLength").alias("intervallength"),
        f.col("blockstarttime"),
        f.col("blockendtime"),
        f.col("NumberIntervals").alias("numberintervals"),
        f.col("starttime"),
        f.col("endtime"),
        f.col("interval_epoch"),
        f.col("int_gatewaycollectedtime"),
        f.col("int_blocksequencenumber"),
        f.col("int_intervalsequencenumber"),
        f.col("channel"),
        f.col("value").alias("aep_raw_value"),
        f.col("value"),
        f.col("uom").alias("aep_raw_uom"),
        f.upper(f.col("aep_uom")).alias("aep_derived_uom"),
        f.col("name_register"),
        f.col("aep_sqi").alias("aep_srvc_qlty_idntfr"),
        f.col("prem_nb").alias("aep_premise_nb"),
        f.col("bill_cnst"),
        f.col("acct_clas_cd").alias("aep_acct_cls_cd"),
        f.col("acct_type_cd").alias("aep_acct_type_cd"),
        f.col("devc_cd").alias("aep_devicecode"),
        f.col("pgm_id_nm").alias("aep_meter_program"),
        f.col("sd").alias("aep_srvc_dlvry_id"),
        f.col("mtr_pnt_nb").alias("aep_mtr_pnt_nb"),
        f.col("tarf_pnt_nb").alias("aep_tarf_pnt_nb"),
        f.col("cmsg_mtr_mult_nb").alias("aep_comp_mtr_mltplr"),
        f.col("mtr_rmvl_ts").alias("aep_mtr_removal_ts"),
        f.col("mtr_inst_ts").alias("aep_mtr_install_ts"),
        f.col("aep_city"),
        f.substring(f.col("aep_zip"), 1, 5).alias("aep_zip"),
        f.col("aep_state"),
        f.lit(HDP_UPDATE_USER).alias("hdp_update_user"),
        f.col("part_date"),
        f.col("value_mltplr_flg"),
        f.substring(f.trim(f.col("MeterName")), -2, 2).alias("aep_meter_bucket")
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
interval_data_files_oh_stg_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- filename: string (nullable = false)
 |-- aep_opco: string (nullable = false)
 |-- serialnumber: string (nullable = true)
 |-- utildeviceid: string (nullable = true)
 |-- macid: string (nullable = true)
 |-- intervallength: string (nullable = true)
 |-- blockstarttime: string (nullable = true)
 |-- blockendtime: string (nullable = true)
 |-- numberintervals: string (nullable = true)
 |-- starttime: string (nullable = true)
 |-- endtime: string (nullable = true)
 |-- interval_epoch: string (nullable = true)
 |-- int_gatewaycollectedtime: string (nullable = true)
 |-- int_blocksequencenumber: string (nullable = true)
 |-- int_intervalsequencenumber: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- aep_raw_value: string (nullable = true)
 |-- value: string (nullable = true)
 |-- aep_raw_uom: string (nullable = true)
 |-- aep_derived_uom: string (nullable = false)
 |-- name_register: string (nullable = true)
 |-- aep_srvc_qlty_idntfr: string (nullable = false)


In [20]:
# ============================================================
# Reference: default.iceberg_interval_data_files_oh_stg_vw
# ============================================================
# Source: scripts/ddl/stg_nonvee.interval_data_files_oh_stg_vw.ddl
# Called by: xfrm_interval_data_files.sh (Line 190)
# Input: interval_data_files_oh_stg_df (41 columns)
# Output: interval_data_files_oh_stg_vw_df (43 columns)

# Transform staged data by adding business logic:
# - Add literals (source, isvirtual flags, timezone, usage_type)
# - Compute aep_service_point from premise/tarf/mtr concatenation
# - Calculate aep_sec_per_intrvl (interval length in seconds)
# - Apply value multiplier logic (value * bill_cnst if flag='Y')
# - Convert endtime to UTC unix timestamp
# - Derive authority and aep_usage_dt from existing columns
# - Add audit timestamps (hdp_insert_dttm, hdp_update_dttm)

interval_data_files_oh_stg_vw_df = (
    interval_data_files_oh_stg_df
    .select(
        f.col("serialnumber"),
        f.lit("nonvee-hes").alias("source"),
        f.col("aep_devicecode"),
        f.lit("N").alias("isvirtual_meter"),
        f.col("interval_epoch").alias("timezoneoffset"),
        f.col("aep_premise_nb"),
        f.when(
            f.col("aep_premise_nb").isNotNull() &
            f.col("aep_tarf_pnt_nb").isNotNull() &
            f.col("aep_mtr_pnt_nb").isNotNull(),
            f.concat(f.col("aep_premise_nb"), f.lit("-"), f.col("aep_tarf_pnt_nb"), f.lit("-"), f.col("aep_mtr_pnt_nb"))
        ).otherwise(f.lit(None)).alias("aep_service_point"),
        f.col("aep_srvc_dlvry_id"),
        f.col("name_register"),
        f.lit("N").alias("isvirtual_register"),
        f.col("aep_derived_uom"),
        f.col("aep_raw_uom"),
        f.col("aep_srvc_qlty_idntfr"),
        f.col("channel").alias("aep_channel_id"),
        (f.col("intervallength").cast("int") * 60).alias("aep_sec_per_intrvl"),
        f.lit(None).cast("string").alias("aep_meter_alias"),
        f.col("aep_meter_program"),
        f.lit("interval").alias("aep_usage_type"),
        f.lit("US/Eastern").alias("aep_timezone_cd"),
        f.col("endtime").alias("endtimeperiod"),
        f.col("starttime").alias("starttimeperiod"),
        f.when(
            f.col("value_mltplr_flg") == "Y",
            f.col("value").cast("float") * f.col("bill_cnst").cast("float")
        ).otherwise(f.col("value").cast("float")).alias("value"),
        f.col("aep_raw_value"),
        f.col("bill_cnst").alias("scalarfloat"),
        f.col("aep_acct_cls_cd"),
        f.col("aep_acct_type_cd"),
        f.col("aep_mtr_pnt_nb"),
        f.col("aep_tarf_pnt_nb"),
        f.col("aep_comp_mtr_mltplr").cast("double").alias("aep_comp_mtr_mltplr"),
        f.unix_timestamp(
            f.concat(f.col("endtime"), f.col("interval_epoch")),
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        ).cast("string").alias("aep_endtime_utc"),
        f.col("aep_mtr_removal_ts"),
        f.col("aep_mtr_install_ts"),
        f.col("aep_city"),
        f.col("aep_zip"),
        f.col("aep_state"),
        f.from_unixtime(f.unix_timestamp(f.current_timestamp())).alias("hdp_insert_dttm"),
        f.from_unixtime(f.unix_timestamp(f.current_timestamp())).alias("hdp_update_dttm"),
        f.col("hdp_update_user"),
        f.substring(f.col("aep_premise_nb"), 1, 2).alias("authority"),
        f.substring(f.col("starttime"), 1, 10).alias("aep_usage_dt"),
        f.lit("new").alias("data_type"),
        f.col("aep_opco"),
        f.col("aep_meter_bucket")
    )
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
interval_data_files_oh_stg_vw_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- serialnumber: string (nullable = true)
 |-- source: string (nullable = false)
 |-- aep_devicecode: string (nullable = true)
 |-- isvirtual_meter: string (nullable = false)
 |-- timezoneoffset: string (nullable = true)
 |-- aep_premise_nb: string (nullable = true)
 |-- aep_service_point: string (nullable = true)
 |-- aep_srvc_dlvry_id: string (nullable = true)
 |-- name_register: string (nullable = true)
 |-- isvirtual_register: string (nullable = false)
 |-- aep_derived_uom: string (nullable = false)
 |-- aep_raw_uom: string (nullable = true)
 |-- aep_srvc_qlty_idntfr: string (nullable = false)
 |-- aep_channel_id: string (nullable = true)
 |-- aep_sec_per_intrvl: integer (nullable = true)
 |-- aep_meter_alias: string (nullable = true)
 |-- aep_meter_program: string (nullable = true)
 |-- aep_usage_type: string (nullable = false)
 |-- aep_timezone_cd: string (nullable = false)
 |-- endtimeperiod: string (nullable = true)
 |-- starttimeperiod: string (nullable = true)
 |-- val

In [22]:
# ============================================================
# Reference: iceberg_catalog.stg_nonvee.interval_data_files_oh_xfrm
# ============================================================
# Source: scripts/xfrm_interval_data_files.sh (Lines 140-191)
#         scripts/ddl/stg_nonvee.interval_data_files_oh_xfrm.ddl
#         scripts/ddl/stg_nonvee.interval_data_files_oh_xfrm_vw.ddl
# Input: interval_data_files_oh_stg_vw_df (43 columns)
# Output: interval_data_files_oh_xfrm_df (50 columns, deduplicated)
#
# Prepare data for Iceberg xfrm table by:
# - Combined both queries like (stg_nonvee.interval_data_files_oh_xfrm.ddl+interval_data_files_oh_xfrm_vw.ddl)
# - Adding 5 NULL columns (toutier, toutiername, aep_billable_ind, aep_data_quality_cd, aep_data_validation)
# - Casting to match DDL types (double, float, timestamp)
# - Deduplication using ROW_NUMBER() partitioned by (serialnumber, endtimeperiod, aep_channel_id, aep_raw_uom)
# ============================================================

from pyspark.sql.window import Window 
from pyspark.sql.functions import lit, col, row_number

dedup_window = Window.partitionBy(
    "serialnumber",
    "endtimeperiod",
    "aep_channel_id",
    "aep_raw_uom"
).orderBy(col("hdp_insert_dttm").desc())

interval_data_files_oh_xfrm_df = (
    interval_data_files_oh_stg_vw_df
    .select(
        col("serialnumber"),
        col("source"),
        col("aep_devicecode"),
        col("isvirtual_meter"),
        col("timezoneoffset"),
        col("aep_premise_nb"),
        col("aep_service_point"),
        col("aep_mtr_install_ts"),
        col("aep_mtr_removal_ts"),
        col("aep_srvc_dlvry_id"),
        col("aep_comp_mtr_mltplr").cast("double"),
        col("name_register"),
        col("isvirtual_register"),
        lit(None).cast("string").alias("toutier"),
        lit(None).cast("string").alias("toutiername"),
        col("aep_srvc_qlty_idntfr"),
        col("aep_channel_id"),
        col("aep_raw_uom"),
        col("aep_sec_per_intrvl").cast("double"),
        col("aep_meter_alias"),
        col("aep_meter_program"),
        lit(None).cast("string").alias("aep_billable_ind"),
        col("aep_usage_type"),
        col("aep_timezone_cd"),
        col("endtimeperiod"),
        col("starttimeperiod"),
        col("value").cast("float"),
        col("aep_raw_value").cast("float"),
        col("scalarfloat").cast("float"),
        lit(None).cast("string").alias("aep_data_quality_cd"),
        lit(None).cast("string").alias("aep_data_validation"),
        col("aep_acct_cls_cd"),
        col("aep_acct_type_cd"),
        col("aep_mtr_pnt_nb"),
        col("aep_tarf_pnt_nb"),
        col("aep_endtime_utc"),
        col("aep_city"),
        col("aep_zip"),
        col("aep_state"),
        col("hdp_update_user"),
        col("hdp_insert_dttm").cast("timestamp"),
        col("hdp_update_dttm").cast("timestamp"),
        col("authority"),
        col("aep_derived_uom"),
        col("data_type"),
        col("aep_opco"),
        col("aep_usage_dt"),
        col("aep_meter_bucket")
    )
    .withColumn("row_num", row_number().over(dedup_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
interval_data_files_oh_xfrm_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- serialnumber: string (nullable = true)
 |-- source: string (nullable = false)
 |-- aep_devicecode: string (nullable = true)
 |-- isvirtual_meter: string (nullable = false)
 |-- timezoneoffset: string (nullable = true)
 |-- aep_premise_nb: string (nullable = true)
 |-- aep_service_point: string (nullable = true)
 |-- aep_mtr_install_ts: string (nullable = true)
 |-- aep_mtr_removal_ts: string (nullable = true)
 |-- aep_srvc_dlvry_id: string (nullable = true)
 |-- aep_comp_mtr_mltplr: double (nullable = true)
 |-- name_register: string (nullable = true)
 |-- isvirtual_register: string (nullable = false)
 |-- toutier: string (nullable = true)
 |-- toutiername: string (nullable = true)
 |-- aep_srvc_qlty_idntfr: string (nullable = false)
 |-- aep_channel_id: string (nullable = true)
 |-- aep_raw_uom: string (nullable = true)
 |-- aep_sec_per_intrvl: double (nullable = true)
 |-- aep_meter_alias: string (nullable = true)
 |-- aep_meter_program: string (nullable = true)
 |-- aep_bi

In [24]:
# ============================================================
# Reference: MERGE INTO usage_nonvee.reading_ivl_nonvee_oh
# ============================================================
# Source: scripts/dml/usage_nonvee.reading_ivl_nonvee.dml
# Input: interval_data_files_oh_xfrm_df
# Target: iceberg_catalog.usage_nonvee.reading_ivl_nonvee_oh
#
# MERGE operation to upsert meter readings:
# - MATCHED: Update existing records
# - NOT MATCHED: Insert new records
# - Keys: serialnumber, endtimeperiod, aep_channel_id, aep_raw_uom,
#         aep_usage_dt, aep_meter_bucket
# ============================================================


# Step 1: Register DataFrame as temp view
interval_data_files_oh_xfrm_df.createOrReplaceTempView("interval_data_files_oh_xfrm_vw")

# Step 2: Get distinct usage dates for partition pruning
usage_dates_df = interval_data_files_oh_xfrm_df.select("aep_usage_dt").distinct()
usage_dates_list = [row.aep_usage_dt for row in usage_dates_df.collect()]
usage_dates_str = ",".join([f"'{d}'" for d in usage_dates_list])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
# Step 1: checkpoint directory
spark.sparkContext.setCheckpointDir("s3://aep-datalake-work-dev/temp/checkpoint/")

# Step 2: Materialize the DataFrame (saves row_number result to disk)
materialized_df = interval_data_files_oh_xfrm_df.checkpoint()

# Step 3: Create temp view from materialized data
materialized_df.createOrReplaceTempView("interval_data_files_oh_xfrm_vw")

# Step 4: Get distinct usage dates for partition pruning
usage_dates_df = materialized_df.select("aep_usage_dt").distinct()
usage_dates_list = [row.aep_usage_dt for row in usage_dates_df.collect()]
usage_dates_str = ",".join([f"'{d}'" for d in usage_dates_list])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
#from pyspark.storagelevel import StorageLevel

#materialized_df = interval_data_files_oh_xfrm_df.persist(StorageLevel.MEMORY_AND_DISK)

#materialized_df.count()

#materialized_df.createOrReplaceTempView("interval_data_files_oh_xfrm_vw")

#usage_dates_df = materialized_df.select("aep_usage_dt").distinct()
#usage_dates_list = [row.aep_usage_dt for row in usage_dates_df.collect()]
#usage_dates_str = ",".join([f"'{d}'" for d in usage_dates_list])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
print(usage_dates_str)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

'2026-02-19'

In [28]:
spark.sql(f"select * from iceberg_catalog.usage_nonvee.reading_ivl_nonvee_oh where aep_usage_dt={usage_dates_str}").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----------+--------------+---------------+--------------+--------------+-------------------+--------------------+------------------+-----------------+-------------------+---------------+------------------+-------+-----------+--------------------+--------------+-----------+------------------+---------------+-----------------+----------------+--------------+---------------+--------------------+--------------------+------+-------------+-----------+-------------------+-------------------+---------------+----------------+--------------+---------------+---------------+----------+-------+---------+---------------+--------------------+-------------------+---------+---------------+--------+------------+----------------+
|serialnumber|    source|aep_devicecode|isvirtual_meter|timezoneoffset|aep_premise_nb|  aep_service_point|  aep_mtr_install_ts|aep_mtr_removal_ts|aep_srvc_dlvry_id|aep_comp_mtr_mltplr|  name_register|isvirtual_register|toutier|toutiername|aep_srvc_qlty_idntfr|aep_c

In [29]:
merge_sql = f"""
MERGE INTO iceberg_catalog.usage_nonvee.reading_ivl_nonvee_oh AS t
USING (
    SELECT
        serialnumber,
        source,
        aep_devicecode,
        isvirtual_meter,
        timezoneoffset,
        aep_premise_nb,
        aep_service_point,
        aep_srvc_dlvry_id,
        name_register,
        isvirtual_register,
        toutier,
        toutiername,
        aep_derived_uom,
        aep_raw_uom,
        aep_srvc_qlty_idntfr,
        aep_channel_id,
        aep_sec_per_intrvl,
        aep_meter_alias,
        aep_meter_program,
        aep_billable_ind,
        aep_usage_type,
        aep_timezone_cd,
        endtimeperiod,
        starttimeperiod,
        value,
        aep_raw_value,
        scalarfloat,
        aep_data_quality_cd,
        aep_data_validation,
        aep_acct_cls_cd,
        aep_acct_type_cd,
        aep_mtr_pnt_nb,
        aep_tarf_pnt_nb,
        aep_comp_mtr_mltplr,
        aep_endtime_utc,
        aep_mtr_removal_ts,
        aep_mtr_install_ts,
        aep_city,
        aep_zip,
        aep_state,
        to_timestamp(hdp_insert_dttm, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as hdp_insert_dttm,
        to_timestamp(hdp_insert_dttm, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as hdp_update_dttm,
        hdp_update_user,
        authority,
        aep_opco,
        aep_usage_dt,
        aep_meter_bucket
    FROM interval_data_files_oh_xfrm_vw
) AS s
ON t.aep_usage_dt IN ({usage_dates_str})
   AND t.aep_usage_dt = s.aep_usage_dt
   AND t.aep_meter_bucket = s.aep_meter_bucket
   AND t.serialnumber = s.serialnumber
   AND t.aep_channel_id = s.aep_channel_id
   AND t.aep_raw_uom = s.aep_raw_uom
   AND t.endtimeperiod = s.endtimeperiod
WHEN MATCHED THEN
    UPDATE SET
        t.aep_srvc_qlty_idntfr = s.aep_srvc_qlty_idntfr,
        t.aep_channel_id = s.aep_channel_id,
        t.aep_sec_per_intrvl = s.aep_sec_per_intrvl,
        t.aep_meter_alias = s.aep_meter_alias,
        t.aep_meter_program = s.aep_meter_program,
        t.aep_usage_type = s.aep_usage_type,
        t.aep_timezone_cd = s.aep_timezone_cd,
        t.endtimeperiod = s.endtimeperiod,
        t.starttimeperiod = s.starttimeperiod,
        t.value = s.value,
        t.aep_raw_value = s.aep_raw_value,
        t.scalarfloat = s.scalarfloat,
        t.aep_acct_cls_cd = s.aep_acct_cls_cd,
        t.aep_acct_type_cd = s.aep_acct_type_cd,
        t.aep_mtr_pnt_nb = s.aep_mtr_pnt_nb,
        t.aep_comp_mtr_mltplr = s.aep_comp_mtr_mltplr,
        t.aep_endtime_utc = s.aep_endtime_utc,
        t.aep_mtr_removal_ts = s.aep_mtr_removal_ts,
        t.aep_mtr_install_ts = s.aep_mtr_install_ts,
        t.aep_city = s.aep_city,
        t.aep_zip = s.aep_zip,
        t.aep_state = s.aep_state,
        t.hdp_update_user = 'info-update',
        t.hdp_update_dttm = s.hdp_update_dttm,
        t.authority = s.authority
WHEN NOT MATCHED THEN
    INSERT (
        serialnumber,
        source,
        aep_devicecode,
        isvirtual_meter,
        timezoneoffset,
        aep_premise_nb,
        aep_service_point,
        aep_srvc_dlvry_id,
        name_register,
        isvirtual_register,
        toutier,
        toutiername,
        aep_derived_uom,
        aep_raw_uom,
        aep_srvc_qlty_idntfr,
        aep_channel_id,
        aep_sec_per_intrvl,
        aep_meter_alias,
        aep_meter_program,
        aep_billable_ind,
        aep_usage_type,
        aep_timezone_cd,
        endtimeperiod,
        starttimeperiod,
        value,
        aep_raw_value,
        scalarfloat,
        aep_data_quality_cd,
        aep_data_validation,
        aep_acct_cls_cd,
        aep_acct_type_cd,
        aep_mtr_pnt_nb,
        aep_tarf_pnt_nb,
        aep_comp_mtr_mltplr,
        aep_endtime_utc,
        aep_mtr_removal_ts,
        aep_mtr_install_ts,
        aep_city,
        aep_zip,
        aep_state,
        hdp_update_user,
        hdp_insert_dttm,
        hdp_update_dttm,
        authority,
        aep_opco,
        aep_usage_dt,
        aep_meter_bucket
    ) VALUES (
        s.serialnumber,
        s.source,
        s.aep_devicecode,
        s.isvirtual_meter,
        s.timezoneoffset,
        s.aep_premise_nb,
        s.aep_service_point,
        s.aep_srvc_dlvry_id,
        s.name_register,
        s.isvirtual_register,
        s.toutier,
        s.toutiername,
        s.aep_derived_uom,
        s.aep_raw_uom,
        s.aep_srvc_qlty_idntfr,
        s.aep_channel_id,
        s.aep_sec_per_intrvl,
        s.aep_meter_alias,
        s.aep_meter_program,
        s.aep_billable_ind,
        s.aep_usage_type,
        s.aep_timezone_cd,
        s.endtimeperiod,
        s.starttimeperiod,
        s.value,
        s.aep_raw_value,
        s.scalarfloat,
        s.aep_data_quality_cd,
        s.aep_data_validation,
        s.aep_acct_cls_cd,
        s.aep_acct_type_cd,
        s.aep_mtr_pnt_nb,
        s.aep_tarf_pnt_nb,
        s.aep_comp_mtr_mltplr,
        s.aep_endtime_utc,
        s.aep_mtr_removal_ts,
        s.aep_mtr_install_ts,
        s.aep_city,
        s.aep_zip,
        s.aep_state,
        'info-insert',
        s.hdp_insert_dttm,
        s.hdp_update_dttm,
        s.authority,
        s.aep_opco,
        s.aep_usage_dt,
        s.aep_meter_bucket
    )
"""
spark.sql(merge_sql)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [30]:
check_df=spark.sql(f"select * from iceberg_catalog.usage_nonvee.reading_ivl_nonvee_oh where aep_usage_dt={usage_dates_str}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [31]:
check_df.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----------+--------------+---------------+--------------+--------------+-------------------+--------------------+------------------+-----------------+-------------------+---------------+------------------+-------+-----------+--------------------+--------------+-----------+------------------+---------------+-----------------+----------------+--------------+---------------+--------------------+--------------------+------+-------------+-----------+-------------------+-------------------+---------------+----------------+--------------+---------------+---------------+----------+-------+---------+---------------+--------------------+-------------------+---------+---------------+--------+------------+----------------+
|serialnumber|    source|aep_devicecode|isvirtual_meter|timezoneoffset|aep_premise_nb|  aep_service_point|  aep_mtr_install_ts|aep_mtr_removal_ts|aep_srvc_dlvry_id|aep_comp_mtr_mltplr|  name_register|isvirtual_register|toutier|toutiername|aep_srvc_qlty_idntfr|aep_c

In [34]:
insert_downstream_sql = """
INSERT INTO iceberg_catalog.xfrm_interval.reading_ivl_nonvee_incr_test
SELECT
    serialnumber,
    source,
    aep_devicecode,
    isvirtual_meter,
    timezoneoffset,
    aep_premise_nb,
    aep_service_point,
    aep_mtr_install_ts,
    aep_mtr_removal_ts,
    aep_srvc_dlvry_id,
    aep_comp_mtr_mltplr,
    name_register,
    isvirtual_register,
    toutier,
    toutiername,
    aep_srvc_qlty_idntfr,
    aep_channel_id,
    aep_raw_uom,
    aep_sec_per_intrvl,
    aep_meter_alias,
    aep_meter_program,
    aep_billable_ind,
    aep_usage_type,
    aep_timezone_cd,
    endtimeperiod,
    starttimeperiod,
    value,
    aep_raw_value,
    scalarfloat,
    aep_data_quality_cd,
    aep_data_validation,
    aep_acct_cls_cd,
    aep_acct_type_cd,
    aep_mtr_pnt_nb,
    aep_tarf_pnt_nb,
    aep_endtime_utc,
    aep_city,
    aep_zip,
    aep_state,
    hdp_update_user,
    hdp_insert_dttm,
    hdp_update_dttm,
    authority,
    aep_usage_dt,
    aep_derived_uom,
    aep_opco,
    date_format(current_timestamp(), 'yyyyMMdd_HHmmss') as run_date
FROM interval_data_files_oh_xfrm_vw
WHERE data_type = 'new' AND aep_opco = 'oh'
"""

spark.sql(insert_downstream_sql)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [36]:
print(date_format)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
name 'date_format' is not defined
Traceback (most recent call last):
NameError: name 'date_format' is not defined



In [44]:
spark.sql("""select * from iceberg_catalog.xfrm_interval.reading_ivl_nonvee_incr_test  group by aep_opco='OH'""").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
[MISSING_AGGREGATION] The non-aggregating expression "serialnumber" is based on columns which are not participating in the GROUP BY clause.
Add the columns or the expression to the GROUP BY, aggregate the expression, or use "any_value(serialnumber)" if you do not care which of the values within a group is returned.;
Aggregate [(aep_opco#2995 = OH)], [serialnumber#2950, source#2951, aep_devicecode#2952, isvirtual_meter#2953, timezoneoffset#2954, aep_premise_nb#2955, aep_service_point#2956, aep_mtr_install_ts#2957, aep_mtr_removal_ts#2958, aep_srvc_dlvry_id#2959, aep_comp_mtr_mltplr#2960, name_register#2961, isvirtual_register#2962, toutier#2963, toutiername#2964, aep_srvc_qlty_idntfr#2965, aep_channel_id#2966, aep_raw_uom#2967, aep_sec_per_intrvl#2968, aep_meter_alias#2969, aep_meter_program#2970, aep_billable_ind#2971, aep_usage_type#2972, aep_timezone_cd#2973, ... 23 more fields]
+- SubqueryAlias iceberg_catalog.xfrm_interval.reading_ivl_nonvee_incr_test
   +